### install the neccessary packages
transformers - To use huggingface transformers "t5_small" model

evaluate - A library for easily evaluating the model

datasets - A library for easily accessing and sharing datasets


In [1]:
!pip install transformers datasets evaluate

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 72.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 KB 41.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.4/81.4 KB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 KB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 94.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 KB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 KB 23.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 KB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 KB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 KB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━

### import:
AutoTokenizer - a generic tokenizer class

files and io - used for uploading the dataset


In [2]:
from google.colab import files
import io
import pandas as pd
from transformers import AutoTokenizer
from datasets import Dataset
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union
import torch
import evaluate
import numpy as np
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
import datasets

### upload the dataset
Medcqa is a dataset of 4183 rows containing medical multiple choice questions and their answer.

In [4]:
uploaded = files.upload()
# read the dev.json file
dev_df = pd.read_json(io.BytesIO(uploaded["dev.json"]), encoding="ISO-8859-1", lines=True)
dev_df = pd.DataFrame(dev_df)

print(len(dev_df))

Saving dev.json to dev.json
4183


rename the 'cop' column to 'label'

In [6]:
dev_df.rename(columns={'cop':'label'}, inplace=True)

### Preproccess the data
copy each question 4 times and add the 4 answers and tokenize each one of them using AutoTokenizer class. truncate text to be no more than the max_length

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [8]:
ending_names = ["opa", "opb", "opc", "opd"]


def preprocess_function(examples):
    first_sentences = [[context] * 4 for context in examples["question"]]
    second_sentences = [examples[end] for end in ending_names] 
    # print(second_sentences)
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(first_sentences, second_sentences, truncation=True)
    return {k: [v[i : i + 4] for i in range(0, len(v), 4)] for k, v in tokenized_examples.items()}

### split the data 
split the data into train and test set.  'label' column -1 because indices start from 0

In [9]:
# use (2800 rows) for training
train_df = dev_df.iloc[400:3200, :]

# use the rest (400 rows) for testing
dev_df = dev_df.iloc[:400, :]
# 'label' column -1 because indices start from 0
train_df["label"]-=1
dev_df["label"]-=1
train_ds = Dataset.from_pandas(train_df)
dev_ds = Dataset.from_pandas(dev_df)

<ipython-input-9-cebe78dc759b>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["label"]-=1
<ipython-input-9-cebe78dc759b>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dev_df["label"]-=1


In [10]:
print(train_ds)
print(dev_ds)

Dataset({
    features: ['question', 'exp', 'label', 'opa', 'opb', 'opc', 'opd', 'subject_name', 'topic_name', 'id', 'choice_type'],
    num_rows: 2800
})
Dataset({
    features: ['question', 'exp', 'label', 'opa', 'opb', 'opc', 'opd', 'subject_name', 'topic_name', 'id', 'choice_type'],
    num_rows: 400
})


map the preprocess function to each row of dataset

In [12]:
tokenized_train_dataset = train_ds.map(preprocess_function, batched=True)
tokenized_dev_dataset = dev_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

### define a DataCollator for multiple choice

In [13]:
@dataclass
class DataCollatorForMultipleChoice:
    """
    Data collator that will dynamically pad the inputs for multiple choice received.
    """

    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch

### use accuracy metric
use accuracy metric to eevaluate the model

In [14]:
accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

### Login to huggingface hub
with a token in order to be able to push the model to the hub after training.

In [15]:
from huggingface_hub import notebook_login

notebook_login()

Token is valid.
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful


### Train the model 
train the model (30 epochs) with specified arguments, tokenizer, evaluzation metric,.. using AutoModelForMultipleChoice class.

In [17]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
training_args = TrainingArguments(
    output_dir="multiple_answer_QA",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=30,
    weight_decay=0.001,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_dev_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMultipleChoice: ['cls.predictions.decoder.weight', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForMultipleChoice from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMultipleChoice from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly

Download file pytorch_model.bin:   0%|          | 15.4k/418M [00:00<?, ?B/s]

Download file runs/Apr08_19-31-56_d028dcf8760c/events.out.tfevents.1680982325.d028dcf8760c.14526.0: 100%|#####…

Download file runs/Apr08_17-11-07_5caa3e28c395/events.out.tfevents.1680973966.5caa3e28c395.168.0: 100%|#######…

Download file runs/Apr08_19-31-56_d028dcf8760c/1680982325.8411887/events.out.tfevents.1680982325.d028dcf8760c.…

...

Clean file training_args.bin:  29%|##8       | 1.00k/3.50k [00:00<?, ?B/s]

Clean file pytorch_model.bin:   0%|          | 1.00k/418M [00:00<?, ?B/s]

/usr/local/lib/python3.9/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.388630,0.230000
2,No log,1.389171,0.255000
3,1.393600,1.386256,0.237500
4,1.393600,1.386751,0.255000
5,1.393600,1.451108,0.260000
6,1.297300,1.673779,0.240000
7,1.297300,1.968358,0.265000
8,1.297300,2.500211,0.280000
9,0.691600,2.987175,0.280000
10,0.691600,3.235435,0.280000


TrainOutput(global_step=5250, training_loss=0.3670563617206755, metrics={'train_runtime': 5232.7374, 'train_samples_per_second': 16.053, 'train_steps_per_second': 1.003, 'total_flos': 1.2282670226358912e+16, 'train_loss': 0.3670563617206755, 'epoch': 30.0})

In [18]:
trainer.push_to_hub()

Several commits (2) will be pushed upstream.
The progress bars may be unreliable.


Upload file pytorch_model.bin:   0%|          | 1.00/418M [00:00<?, ?B/s]

Upload file runs/Apr09_15-05-15_de49c45459a8/events.out.tfevents.1681052816.de49c45459a8.1474.0:   0%|        …

To https://huggingface.co/TaniyaHaghighi/multiple_answer_QA
   7eee930..b958cb3  main -> main

   7eee930..b958cb3  main -> main

To https://huggingface.co/TaniyaHaghighi/multiple_answer_QA
   b958cb3..ff86fda  main -> main

   b958cb3..ff86fda  main -> main



'https://huggingface.co/TaniyaHaghighi/multiple_answer_QA/commit/b958cb32ea85c8ad3b1a8dd768531d7857a054ce'

### Inference 
an example to test the model 

In [20]:
prompt = "Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma"
candidate1 = "Hyperplasia"
candidate2 ="Hyperophy"
candidate3 ="Atrophy"
candidate4 ="Dyplasia"


In [21]:
tokenizer = AutoTokenizer.from_pretrained("multiple_answer_QA")
inputs = tokenizer([[prompt, candidate1], [prompt, candidate2], [prompt, candidate3], [prompt, candidate4]], return_tensors="pt", padding=True)
labels = torch.tensor(0).unsqueeze(0)

In [22]:
model = AutoModelForMultipleChoice.from_pretrained("multiple_answer_QA")
outputs = model(**{k: v.unsqueeze(0) for k, v in inputs.items()}, labels=labels)
logits = outputs.logits
print(logits)

tensor([[-0.1992, -0.1977, -0.1885, -0.1875]], grad_fn=<ViewBackward0>)


In [23]:
predicted_class = logits.argmax().item()
print(predicted_class)

3
